# Chapter 6: Generative AI

**Welcome to Chapter 6**. This notebook contains the listings for Chapter 6, which explains the fundamentals of Deep Learning in PyTorch.

# Listing 6-1: Training a Variational Autoencoder to Generate 3D Mesh Variations

This code implements a simple VAE that learns a latent representation of perturbed icosphere meshes. The encoder maps flattened vertex coordinates to latent distribution parameters, the decoder reconstructs the mesh, and sampling from the latent distribution generates controlled geometric variations.

In [ ]:
# --------------------------------------------------
# Step 1: Install dependencies
# --------------------------------------------------
!pip install -q trimesh matplotlib

# --------------------------------------------------
# Step 2: Imports and Device Setup
# --------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import trimesh
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Step 3: Crumpled Paper Dataset (Perturbed Spheres)
# --------------------------------------------------
class SphereMeshDataset(Dataset):
    def __init__(self, num_samples=300, subdivisions=2):
        self.mesh_template = trimesh.creation.icosphere(subdivisions=subdivisions)
        self.num_samples = num_samples
        self.faces = self.mesh_template.faces

    def __len__(self):
        return self.num_samples

    def __getitem__(self, _):
        scale = np.random.uniform(0.04, 0.09)
        perturbation = np.random.normal(scale=scale, size=self.mesh_template.vertices.shape)
        verts = self.mesh_template.vertices + perturbation
        return torch.tensor(verts.flatten(), dtype=torch.float32)

# --------------------------------------------------
# Step 4: VAE Architecture and Reparameterization
# --------------------------------------------------
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU()
        )
        self.fc_mu = nn.Linear(64, latent_dim)
        self.fc_logvar = nn.Linear(64, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
            nn.Linear(128, 256), nn.ReLU(),
            nn.Linear(256, input_dim)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z)
        return recon, mu, logvar

# --------------------------------------------------
# Step 5: Loss Function
# --------------------------------------------------
def vae_loss(recon, x, mu, logvar):
    recon_loss = F.mse_loss(recon, x, reduction='sum')
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl

# --------------------------------------------------
# Step 6: Training the VAE
# --------------------------------------------------
dataset = SphereMeshDataset()
input_dim = dataset[0].numel()
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
vae = VAE(input_dim=input_dim, latent_dim=16).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(500):
    total = 0
    for batch in dataloader:
        batch = batch.to(device)
        recon, mu, logvar = vae(batch)
        loss = vae_loss(recon, batch, mu, logvar)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total:.2f}")

# --------------------------------------------------
# Step 7: Inference and Sampling
# --------------------------------------------------
vae.eval()
with torch.no_grad():
    faces = dataset.mesh_template.faces
    example = next(iter(dataloader))[0].unsqueeze(0).to(device)
    recon, mu, logvar = vae(example)

    var1 = vae.decoder(vae.reparameterize(mu, logvar))
    var2 = vae.decoder(vae.reparameterize(mu, logvar))
    var3 = vae.decoder(vae.reparameterize(mu, logvar))

    def unflatten(tensor): return tensor.cpu().numpy().reshape(-1, 3)

    original_verts = unflatten(example)
    recon_verts = unflatten(recon)
    var1_verts = unflatten(var1)
    var2_verts = unflatten(var2)
    var3_verts = unflatten(var3)

# --------------------------------------------------
# Step 8: Visualization
# --------------------------------------------------
def plot_meshes(verts_list, faces, titles):
    fig, axs = plt.subplots(1, 5, figsize=(20,5), subplot_kw={'projection': '3d'})
    paper_yellow = np.array([1.0, 1.0, 0.85])

    for ax, verts, title in zip(axs, verts_list, titles):
        mesh = Poly3DCollection(verts[faces], facecolors=paper_yellow, edgecolors='black', linewidths=0.2, alpha=1.0)
        ax.add_collection3d(mesh)
        ax.set_xlim(-1.2, 1.2)
        ax.set_ylim(-1.2, 1.2)
        ax.set_zlim(-1.2, 1.2)
        ax.axis('off')
        ax.view_init(elev=20, azim=30)
        ax.set_title(title, pad=5, fontsize=22)

    plt.show()

plot_meshes(
    [original_verts, recon_verts, var1_verts, var2_verts, var3_verts],
    faces,
    ["Original", "Reconstruction", "Variation 1", "Variation 2", "Variation 3"]
)

# Listing 6-2: Generating Realistic Faces with StyleGAN2-ADA

This example uses a pretrained StyleGAN2-ADA model to generate photorealistic synthetic faces from random seeds. The notebook clones the official repository, downloads pretrained weights, and runs the generator to produce multiple face images.

In [ ]:
# --------------------------------------------------
# Step 1. Clone StyleGAN2-ADA and Install Dependencies
# --------------------------------------------------
!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git
%cd stylegan2-ada-pytorch

# --------------------------------------------------
# Step 2. Install dependencies
# --------------------------------------------------
import random
!pip install ninja pyspng imageio-ffmpeg==0.4.3

# --------------------------------------------------
# Step 3. Download Pretrained FFHQ Model (Faces)
# --------------------------------------------------
!mkdir pretrained
!wget -O pretrained/ffhq.pkl https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl

# --------------------------------------------------
# Step 4. Let Python choose random seeds for you
# --------------------------------------------------
a = random.randint(0, 7000)
b = a + 11 # Generate 1 + 11 = 12 faces

# --------------------------------------------------
# Step 5. Generate Faces from Pretrained Model
# --------------------------------------------------
!python generate.py \
  --outdir=generated_faces \
  --trunc=0.7 \
  --seeds={a}-{b} \
  --network=pretrained/ffhq.pkl

# --------------------------------------------------
# Step 6. View the Generated Faces
# --------------------------------------------------
from IPython.display import Image, display
import os

for filename in sorted(os.listdir('generated_faces')):
    if filename.endswith('.png'):
       display(Image(filename=f'generated_faces/{filename}'))


# Listing 6-3: Training a GAN on the MNIST Dataset

This example implements adversarial training between a generator and discriminator to synthesize handwritten digits. The generator learns to produce digit images from random noise, while the discriminator learns to distinguish generated images from real MNIST samples.

In [ ]:
# --------------------------------------------------
# Step 1: Install dependencies
# --------------------------------------------------
!pip install -q torch torchvision matplotlib

# --------------------------------------------------
# Step 2: Imports, device, and reproducibility
# --------------------------------------------------
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --------------------------------------------------
# Step 3: Data (MNIST normalized to [-1, 1])
# --------------------------------------------------
batch_size = 128
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# --------------------------------------------------
# Step 4: Model definitions
# --------------------------------------------------
z_dim = 100
img_dim = 28 * 28

class Generator(nn.Module):
    def __init__(self, z_dim, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, img_dim),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

G = Generator(z_dim, img_dim).to(device)
D = Discriminator(img_dim).to(device)

# --------------------------------------------------
# Step 5: Loss and optimizers
# --------------------------------------------------
criterion = nn.BCELoss()
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

# --------------------------------------------------
# Step 6: Training loop
# --------------------------------------------------
epochs = 20
fixed_z = torch.randn(64, z_dim, device=device)

d_losses = []
g_losses = []

for epoch in range(1, epochs + 1):

    d_running = 0.0
    g_running = 0.0
    n_batches = 0

    for real, _ in train_loader:
        n_batches += 1
        real = real.to(device).view(-1, img_dim)
        bsz = real.size(0)

        real_labels = torch.ones(bsz, 1, device=device)
        fake_labels = torch.zeros(bsz, 1, device=device)

        # -----------------------------
        # Step 6a: Train Discriminator
        # -----------------------------

        opt_D.zero_grad(set_to_none=True)

        # 6a.i Real samples
        real_pred = D(real)
        d_loss_real = criterion(real_pred, real_labels)

        # 6a.ii Fake samples
        z = torch.randn(bsz, z_dim, device=device)
        fake = G(z)
        fake_pred = D(fake.detach())
        d_loss_fake = criterion(fake_pred, fake_labels)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        opt_D.step()

        # -----------------------------
        # Step 6b: Train Generator
        # -----------------------------

        opt_G.zero_grad(set_to_none=True)

        z = torch.randn(bsz, z_dim, device=device)
        fake = G(z)
        fake_pred = D(fake)

        # Non-saturating objective
        g_loss = criterion(fake_pred, real_labels)

        g_loss.backward()
        opt_G.step()

        d_running += d_loss.item()
        g_running += g_loss.item()

    d_epoch = d_running / n_batches
    g_epoch = g_running / n_batches

    d_losses.append(d_epoch)
    g_losses.append(g_epoch)

    print(f"Epoch {epoch}/{epochs} | D Loss: {d_epoch:.4f} | G Loss: {g_epoch:.4f}")

    # --------------------------------------------------
    # Step 7: Save sample grid
    # --------------------------------------------------
    with torch.no_grad():
        G.eval()
        samples = G(fixed_z).view(-1, 1, 28, 28)
        grid = utils.make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
        utils.save_image(grid, f"epoch_{epoch:02d}.png")

# --------------------------------------------------
# Step 8: Plot losses (Figure 6-4)
# --------------------------------------------------
epochs_arr = np.arange(1, epochs + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_arr, d_losses, marker="o", linestyle="None", label="Discriminator (D)")
plt.plot(epochs_arr, g_losses, marker="x", linestyle="None", label="Generator (G)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Listing 6-4: Fine-Tuning a GAN from Circles to Slashes

The following code demonstrates GAN fine-tuning by first training a generator on synthetic circle images and then adapting it to generate slashes. The example illustrates transfer learning, selective layer freezing, and differential learning rates.


In [ ]:
# --------------------------------------------------
# Step 1: Imports and Device Setup
# --------------------------------------------------
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

# --------------------------------------------------
# Step 2: Custom Shape Dataset
# --------------------------------------------------
class ShapeDataset(Dataset):
    def __init__(self, count=1000, shape="circle"):
        self.images = []

        for _ in range(count):
            img = Image.new("L", (28, 28), color=0)
            draw = ImageDraw.Draw(img)

            if shape == "circle":
                r = 8
                center = (14, 14)
                box = [center[0] - r, center[1] - r,
                       center[0] + r, center[1] + r]
                draw.ellipse(box, fill=255)

            elif shape == "slash":
                draw.line([5, 23, 23, 5], fill=255, width=3)

            # Normalize pixels to [-1, 1] to match Tanh output
            array = (np.array(img, dtype=np.float32) / 127.5) - 1.0
            self.images.append(torch.tensor(array).unsqueeze(0))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.images[idx]

# --------------------------------------------------
# Step 3: Generator and Discriminator
# --------------------------------------------------
Z_DIM = 100

class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# --------------------------------------------------
# Step 4: Sample Visualization Helper
# --------------------------------------------------
def show_samples(generator, title, n=16):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(n, Z_DIM, device=device)
        samples = generator(z).cpu()

    fig, axs = plt.subplots(2, 8, figsize=(10, 3))
    for i, ax in enumerate(axs.flat):
        img = (samples[i, 0] + 1) / 2  # map [-1, 1] -> [0, 1]
        ax.imshow(img, cmap="gray")
        ax.axis("off")

    plt.suptitle(title)
    plt.show()
    generator.train()

# --------------------------------------------------
# Step 5: Training Function
# --------------------------------------------------
def train_gan(
    generator,
    discriminator,
    dataloader,
    epochs,
    title_prefix="",
    lr=2e-4,
    freeze_g_layers: int = 0,
    use_differential_g_lr: bool = False,
    g_lr_low: float = 1e-5,
    g_lr_high: float = 1e-4,
):
    """
    freeze_g_layers:
        Number of early modules in generator.net to freeze (0 = no freezing).
        This is a simple control knob for the 'freeze early layers' idea.

    use_differential_g_lr:
        If True, use different learning rates for earlier vs later parts of G.
        This matches the "Differential Learning Rates" tip in the text.
    """
    criterion = nn.BCELoss()

    # ------------------------------------------------
    # Alternative Step 5A: Selective Fine-Tuning (Freeze early G layers)
    # ------------------------------------------------
    # Freeze the first `freeze_g_layers` modules inside generator.net.
    # Example: freeze_g_layers=2 freezes the first Linear+ReLU pair.
    if freeze_g_layers > 0:
        modules = list(generator.net.children())
        for m in modules[:freeze_g_layers]:
            for p in m.parameters():
                p.requires_grad = False

    # ------------------------------------------------
    # Step 5B: Optimizers
    # ------------------------------------------------
    # Default: single learning rate for all generator parameters
    if not use_differential_g_lr:
        opt_G = optim.Adam(
            filter(lambda p: p.requires_grad, generator.parameters()),
            lr=lr,
            betas=(0.5, 0.999),
        )
    else:
        # ------------------------------------------------
        # Alternative Step 5B (Option): Differential Learning Rates for G
        # ------------------------------------------------
        # Split generator.net into "earlier" and "later" modules.
        # The last two modules are Linear(256->784) and Tanh in this example.
        g_children = list(generator.net.children())
        early_modules = g_children[:-2]
        late_modules = g_children[-2:]

        early_params = []
        for m in early_modules:
            early_params += [p for p in m.parameters() if p.requires_grad]

        late_params = []
        for m in late_modules:
            late_params += [p for p in m.parameters() if p.requires_grad]

        opt_G = optim.Adam(
            [
                {"params": early_params, "lr": g_lr_low},
                {"params": late_params, "lr": g_lr_high},
            ],
            betas=(0.5, 0.999),
        )

    opt_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    # ------------------------------------------------
    # Step 5C: Training Loop
    # ------------------------------------------------
    for epoch in range(epochs):
        for real_imgs in dataloader:
            real_imgs = real_imgs.to(device)
            batch = real_imgs.size(0)

            real_labels = torch.ones(batch, 1, device=device)
            fake_labels = torch.zeros(batch, 1, device=device)

            # -------------------
            # Step 5C.i: Discriminator update
            # -------------------
            z = torch.randn(batch, Z_DIM, device=device)
            fake_imgs = generator(z)

            loss_D = (
                criterion(discriminator(real_imgs), real_labels) +
                criterion(discriminator(fake_imgs.detach()), fake_labels)
            )

            opt_D.zero_grad()
            loss_D.backward()
            opt_D.step()

            # -------------------
            # Step 5C.ii: Generator update (non-saturating objective)
            # -------------------
            loss_G = criterion(discriminator(fake_imgs), real_labels)

            opt_G.zero_grad()
            loss_G.backward()
            opt_G.step()

        print(
            f"{title_prefix} Epoch {epoch+1}: "
            f"Loss_D={loss_D.item():.4f}, "
            f"Loss_G={loss_G.item():.4f}"
        )

        show_samples(generator, f"{title_prefix} Epoch {epoch+1}")

# --------------------------------------------------
# Step 6: Train on Circles
# --------------------------------------------------
G = Generator().to(device)
D = Discriminator().to(device)

circle_loader = DataLoader(
    ShapeDataset(shape="circle"),
    batch_size=64,
    shuffle=True
)

train_gan(
    G,
    D,
    circle_loader,
    epochs=246,
    title_prefix="CIRCLE"
)

# --------------------------------------------------
# Step 7: Fine-Tune on Slashes
# --------------------------------------------------
slash_loader = DataLoader(
    ShapeDataset(shape="slash"),
    batch_size=64,
    shuffle=True
)

train_gan(
    G,
    D,
    slash_loader,
    epochs=100,
    title_prefix="FINE-TUNE (SLASHES)",
    lr=1e-4,                 # reduced LR for stability
    freeze_g_layers=2,        # Alternative Step: freeze early G layers (optional)
    use_differential_g_lr=False  # set True to enable differential G learning rates
)

# Listing 6-5: Training and Sampling a Simple Diffusion Model on 1D Signals

This example builds a diffusion model from scratch that learns to generate sine-like signals. The example shows forward noising, neural network noise prediction, and reverse diffusion sampling to produce new signals from random noise.

In [ ]:
# --------------------------------------------------
# Step 1: Install and Import Libraries
# --------------------------------------------------
# Install PyTorch and matplotlib for model training and visualization
!pip install -q torch matplotlib

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Select compute device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# Step 2: Create a Synthetic Dataset of Sine Waves
# --------------------------------------------------
class SineDataset(Dataset):
    """
    Generates synthetic sine waves with random amplitude, frequency, and phase.
    Each query produces a fresh sine wave variant for training the diffusion model.
    """
    def __init__(self, n=20000, length=128):
        self.n = n
        self.length = length
        self.x = torch.linspace(0, 1, steps=length)

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Random amplitude between 0.5 and 1.5
        amp = torch.empty(1).uniform_(0.5, 1.5).item()
        # Random frequency between 1.0 and 6.0 Hz
        freq = torch.empty(1).uniform_(1.0, 6.0).item()
        # Random phase shift between 0 and 2π
        phase = torch.empty(1).uniform_(0, 2 * math.pi).item()
        # Generate sine wave with random parameters
        y = amp * torch.sin(2 * math.pi * freq * self.x + phase)
        # Return shape [1, length] for 1D convolution compatibility
        return y.unsqueeze(0)

# --------------------------------------------------
# Step 3: Define the Diffusion Noise Schedule
# --------------------------------------------------
# Number of diffusion timesteps (T=200 means 200 progressive noise levels)
T = 200

# Beta schedule: controls how much noise is added at each timestep
# Linearly increases from 1e-4 to 0.02 across T steps
betas = torch.linspace(1e-4, 0.02, T, device=device)

# Alpha values: proportion of signal retained at each step
alphas = 1.0 - betas

# Cumulative product of alphas: used to jump directly to any timestep
# alpha_bar[t] represents total signal retention from step 0 to step t
alpha_bars = torch.cumprod(alphas, dim=0)

# --------------------------------------------------
# Step 4: Implement Forward Diffusion
# --------------------------------------------------
def q_sample(x0, t, noise=None):
    """
    Forward diffusion: adds noise to clean signal x0 at timestep t.
    This simulates the progressive corruption process that the model will learn to reverse.

    Args:
        x0: Clean signal [B, 1, L] where B=batch, L=length
        t: Timestep indices [B] in range [0, T-1]
        noise: Optional pre-generated noise (if None, creates new random noise)

    Returns:
        xt: Noisy signal at timestep t
        noise: The noise that was added
    """
    if noise is None:
        noise = torch.randn_like(x0)
    # Reshape alpha_bar for broadcasting
    a_bar = alpha_bars[t].view(-1, 1, 1)
    # Apply noise according to diffusion formula: xt = sqrt(alpha_bar) * x0 + sqrt(1 - alpha_bar) * noise
    xt = torch.sqrt(a_bar) * x0 + torch.sqrt(1 - a_bar) * noise
    return xt, noise

# --------------------------------------------------
# Step 5: Define the Noise-Prediction Network
# --------------------------------------------------
class Tiny1DDenoiser(nn.Module):
    """
    Neural network that predicts the noise present in a corrupted signal.
    Uses 1D convolutions and timestep embeddings to denoise signals.
    """
    def __init__(self, length=128, time_dim=64):
        super().__init__()
        # Timestep embedding: converts timestep integer to learned representation
        self.time_emb = nn.Sequential(
            nn.Embedding(T, time_dim),
            nn.Linear(time_dim, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim),
        )

        # Convolutional layers for processing the noisy signal
        self.conv1 = nn.Conv1d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv1d(64, 32, 3, padding=1)
        self.conv4 = nn.Conv1d(32, 1, 3, padding=1)

        # Linear layer to project timestep embedding to match conv2 output
        self.fc_t = nn.Linear(time_dim, 64)

    def forward(self, x, t):
        """
        Args:
            x: Noisy signal [B, 1, L]
            t: Timestep indices [B]

        Returns:
            Predicted noise [B, 1, L]
        """
        # Process signal through first two conv layers
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))

        # Embed timestep and add to feature map (timestep conditioning)
        te = self.time_emb(t)
        te = self.fc_t(te).unsqueeze(-1)
        h = h + te

        # Process through remaining conv layers to predict noise
        h = F.relu(self.conv3(h))
        return self.conv4(h)

# --------------------------------------------------
# Step 6: Train the Model
# --------------------------------------------------
print("Training 1D diffusion model on sine waves...")
print("=" * 60)

# --------------------------------------------------
# Step 6a: Prepare dataset and model
# --------------------------------------------------
# Create dataset of 20,000 synthetic sine waves
dataset = SineDataset(n=20000, length=128)
loader = DataLoader(dataset, batch_size=256, shuffle=True, num_workers=0)

# Initialize model and optimizer
model = Tiny1DDenoiser().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

# --------------------------------------------------
# Step 6b: Training loop
# --------------------------------------------------
# Train for 100 epochs
model.train()
for epoch in range(100):
    total = 0.0

    for x0 in loader:
        x0 = x0.to(device)

        # Sample a random timestep for each example in the batch
        t = torch.randint(0, T, (x0.size(0),), device=device)

        # Apply forward diffusion to corrupt the clean signal
        xt, noise = q_sample(x0, t)

        # Model predicts the noise that was added
        pred = model(xt, t)

        # Loss: compare predicted noise to actual noise
        loss = F.mse_loss(pred, noise)

        # Backpropagation and parameter update
        opt.zero_grad()
        loss.backward()
        opt.step()

        total += loss.item()

    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}: loss={total / len(loader):.4f}")

print("Training complete!")

# --------------------------------------------------
# Step 7: Sample New Signals (Reverse Diffusion)
# --------------------------------------------------
@torch.no_grad()
def p_sample_loop(model, shape):
    """
    Reverse diffusion sampler: generates new signals by progressively denoising pure noise.
    Starts with random noise and iteratively removes predicted noise at each timestep.

    Args:
        model: Trained denoiser network
        shape: Desired output shape [B, 1, L]

    Returns:
        Generated clean signals [B, 1, L]
    """
    model.eval()
    # Start from pure random noise
    x = torch.randn(shape, device=device)

    # Iterate backwards through all timesteps (T-1 down to 0)
    for t in reversed(range(T)):
        # Create timestep tensor for batch
        tt = torch.full((shape[0],), t, device=device, dtype=torch.long)

        # Get noise schedule parameters for this timestep
        a = alphas[t]
        a_bar = alpha_bars[t]
        b = betas[t]

        # Predict the noise currently present in x
        eps = model(x, tt)

        # Compute the reverse-process mean estimate
        # This removes the predicted noise from x
        coef1 = 1 / torch.sqrt(a)
        coef2 = b / torch.sqrt(1 - a_bar)
        mean = coef1 * (x - coef2 * eps)

        # Add stochasticity except at the final step (t=0)
        # This randomness is essential for generating diverse samples
        if t > 0:
            z = torch.randn_like(x)
            sigma = torch.sqrt(b)
            x = mean + sigma * z
        else:
            # At final step, use mean without added noise
            x = mean

    return x

# Generate 8 sample signals using reverse diffusion
samples = p_sample_loop(model, shape=(8, 1, 128)).cpu()

# --------------------------------------------------
# Step 8: Visualize Results
# --------------------------------------------------
print("\nCreating visualization...")

# --------------------------------------------------
# Step 8a: Prepare data for visualization
# --------------------------------------------------
# Get one clean sine wave from dataset
original = dataset[0].to(device).unsqueeze(0)

# Create progressively noisier versions using forward diffusion
# Shows what happens at different timesteps: t=0, 50, 100, 150, 199
timesteps = [0, 50, 100, 150, 199]
noisy_waves = []
for t_val in timesteps:
    t = torch.tensor([t_val], device=device)
    noisy, _ = q_sample(original, t)
    noisy_waves.append(noisy)

# Generate several new samples using reverse diffusion
num_samples = 5
generated = p_sample_loop(model, shape=(num_samples, 1, 128))

# --------------------------------------------------
# Step 8b: Create comprehensive visualization grid
# --------------------------------------------------
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 6, hspace=0.4, wspace=0.3)

# --------------------------------------------------
# Step 8c: Plot original signal
# --------------------------------------------------
ax_orig = fig.add_subplot(gs[0, :2])
ax_orig.plot(original[0, 0].cpu().numpy(), linewidth=2, color="blue")
ax_orig.set_title("Original Sine Wave", fontsize=14, fontweight="bold")
ax_orig.set_xlabel("Time")
ax_orig.set_ylabel("Amplitude")
ax_orig.grid(True, alpha=0.3)

# --------------------------------------------------
# Step 8d: Plot progressively noisier versions from forward diffusion
# --------------------------------------------------
for i, (t_val, noisy) in enumerate(zip(timesteps[1:], noisy_waves[1:])):
    ax = fig.add_subplot(gs[0, i + 2])
    ax.plot(noisy[0, 0].cpu().numpy(), linewidth=1.5, alpha=0.8)
    ax.set_title(f"Noise t={t_val}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.set_ylabel("Amplitude")

# --------------------------------------------------
# Step 8e: Plot highly noisy input (starting point for reverse diffusion)
# --------------------------------------------------
ax_noise = fig.add_subplot(gs[1, 0])
ax_noise.plot(noisy_waves[-1][0, 0].cpu().numpy(), linewidth=1.5, alpha=0.8, color="gray")
ax_noise.set_title("Highly Noisy Input", fontsize=12, fontweight="bold")
ax_noise.set_xlabel("Time")
ax_noise.set_ylabel("Amplitude")
ax_noise.grid(True, alpha=0.3)

# --------------------------------------------------
# Step 8f: Add arrow showing denoising transition
# --------------------------------------------------
ax_arrow = fig.add_subplot(gs[1, 1])
ax_arrow.text(0.5, 0.5, "→\nDenoise\n→", ha="center", va="center",
              fontsize=16, fontweight="bold", transform=ax_arrow.transAxes)
ax_arrow.axis("off")

# --------------------------------------------------
# Step 8g: Plot individual generated samples
# --------------------------------------------------
colors = ["red", "green", "purple", "orange", "brown"]
for i in range(num_samples):
    if i < 4:
        ax = fig.add_subplot(gs[1, i + 2])
    else:
        ax = fig.add_subplot(gs[2, 1])

    ax.plot(generated[i, 0].cpu().numpy(), linewidth=2, alpha=0.9, color=colors[i])
    ax.set_title(f"Generated #{i+1}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time")
    if i == 0 or i == 4:
        ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)

# --------------------------------------------------
# Step 8h: Create overlay plot comparing all generated samples
# --------------------------------------------------
ax_overlay = fig.add_subplot(gs[2, 2:5])
for i in range(num_samples):
    ax_overlay.plot(generated[i, 0].cpu().numpy(), linewidth=2,
                    alpha=0.7, color=colors[i], label=f"Gen #{i+1}")
ax_overlay.plot(original[0, 0].cpu().numpy(), linewidth=3,
                color="blue", linestyle="--", alpha=0.5, label="Original")
ax_overlay.set_title("Generated Samples Overlaid", fontsize=12, fontweight="bold")
ax_overlay.set_xlabel("Time")
ax_overlay.set_ylabel("Amplitude")
ax_overlay.legend(loc="upper right", fontsize=8)
ax_overlay.grid(True, alpha=0.3)

# --------------------------------------------------
# Step 8i: Add statistics summary panel
# --------------------------------------------------
ax_stats = fig.add_subplot(gs[2, 5])
ax_stats.axis("off")
stats_text = (
    "Statistics:\n\n"
    f"Training epochs: 100\n"
    f"Diffusion steps: {T}\n"
    f"Signal length: 128\n"
    f"Generated samples: {num_samples}\n\n"
    "Each generated signal\n"
    "differs because reverse\n"
    "diffusion is stochastic."
)
ax_stats.text(0.1, 0.9, stats_text, fontsize=10, verticalalignment="top",
              transform=ax_stats.transAxes, family="monospace")

# --------------------------------------------------
# Step 8j: Finalize and save visualization
# --------------------------------------------------
plt.suptitle("1D Diffusion Model: Forward Noising and Reverse Sampling",
             fontsize=16, fontweight="bold", y=0.98)

plt.savefig("sine_diffusion_demo.png", dpi=150, bbox_inches="tight")
print("Saved visualization to 'sine_diffusion_demo.png'")
plt.show()


# Listing 6-6: Generating Images from Text with Stable Diffusion

This example uses a pretrained Stable Diffusion v1.5 pipeline to generate multiple images from a text prompt. The notebook loads the diffusion model, applies prompt conditioning, and visualizes several generated variations.

In [ ]:
# --------------------------------------------------
# Step 1. Install and Import Libraries
# --------------------------------------------------
!pip install -q diffusers transformers accelerate scipy safetensors

import torch
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
import matplotlib.pyplot as plt

# --------------------------------------------------
# Step 2. Load the Pretrained Diffusion Pipeline
# --------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=dtype
).to(device)

# Use a sharper sampler than the default
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config)

# Memory optimizations (helpful on smaller GPUs)
pipe.enable_attention_slicing()

# --------------------------------------------------
# Step 3. Read a Text Prompt
# --------------------------------------------------
prompt = input("Enter description: ")

# Optional negative prompt to reduce artifacts
negative_prompt = "blurry, distorted, low quality, bad anatomy"

# --------------------------------------------------
# Step 4. Generate Multiple Image Variations
# --------------------------------------------------
num_images = 4

result = pipe(
    prompt=[prompt] * num_images,
    negative_prompt=[negative_prompt] * num_images,
    num_inference_steps=40,      # more steps = sharper images
    guidance_scale=7.5,          # prompt adherence
    height=512,                  # SD 1.5 native resolution
    width=512
)

images = result.images

# --------------------------------------------------
# Step 5. Display the Images in a Grid
# --------------------------------------------------
cols = 2
rows = (num_images + cols - 1) // cols

fig, axs = plt.subplots(rows, cols, figsize=(10, 10), dpi=120)

# Normalize axs structure
if rows == 1 and cols == 1:
    axs = [[axs]]
elif rows == 1:
    axs = [axs]
elif cols == 1:
    axs = [[ax] for ax in axs]

for i, img in enumerate(images):
    ax = axs[i // cols][i % cols]
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"Variation {i+1}")

# Turn off unused axes
for j in range(i + 1, rows * cols):
    axs[j // cols][j % cols].axis("off")

plt.tight_layout()
plt.show()

# Listing 6-7: Forecasting Airline Passenger Counts with an Autoregressive Model

This code uses a simple autoregressive neural network to predict future monthly airline passenger counts from prior observations. The example builds sliding-window sequences from a historical time series, trains the model to predict the next value, and compares predictions against the held-out test portion.

In [ ]:
# --------------------------------------------------
# Step 1. Install and Import Libraries
# --------------------------------------------------
# Install dependencies (only needed in environments like Colab)
!pip install -q seaborn pandas matplotlib

import torch
import torch.nn as nn
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# --------------------------------------------------
# Step 2. Load and Normalize the Dataset
# --------------------------------------------------
df = sns.load_dataset("flights").dropna()

df['passengers'] = (
    (df['passengers'] - df['passengers'].min()) /
    (df['passengers'].max() - df['passengers'].min())
)

data = df['passengers'].values


# --------------------------------------------------
# Step 3. Create Sliding Window Sequences
# --------------------------------------------------
SEQ_LEN = 12

def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = create_sequences(data, SEQ_LEN)


# --------------------------------------------------
# Step 4. Split Data Into Training and Test Sets
# --------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)


# --------------------------------------------------
# Step 5. Define the Autoregressive Neural Network
# --------------------------------------------------
class ARModel(nn.Module):

    def __init__(self, input_size, hidden_size):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x):
        return self.net(x)

model = ARModel(input_size=SEQ_LEN, hidden_size=16)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


# --------------------------------------------------
# Step 6. Train the Model
# --------------------------------------------------
for epoch in range(100):

    model.train()

    optimizer.zero_grad()

    pred = model(X_train).squeeze()

    loss = criterion(pred, y_train)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/100 - Loss: {loss.item():.4f}")


# --------------------------------------------------
# Step 7. Evaluate the Model
# --------------------------------------------------
model.eval()

with torch.no_grad():
    preds = model(X_test).squeeze().numpy()


# --------------------------------------------------
# Step 8. Undo Normalization
# --------------------------------------------------
y_test_np = y_test.numpy()

min_passengers = df['passengers'].min()
max_passengers = df['passengers'].max()

y_true_rescaled = y_test_np * (max_passengers - min_passengers) + min_passengers
preds_rescaled = preds * (max_passengers - min_passengers) + min_passengers


# --------------------------------------------------
# Step 9. Visualize Predictions
# --------------------------------------------------
plt.figure(figsize=(10,5))

plt.plot(y_true_rescaled, label="True Passengers", marker="o")
plt.plot(preds_rescaled, label="Predicted Passengers", marker="x")

plt.title("Forecasting Airline Passengers")
plt.xlabel("Time Index")
plt.ylabel("Passenger Count")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# Listing 6-8: Generating a Detailed Summary Using a Sequence-to-Sequence Transformer

This example demonstrates summarization with an instruction-tuned FLAN-T5 model. The encoder reads a paragraph about the Amazon Rainforest, and the decoder generates a concise summary based on that encoded representation.

In [ ]:
# --------------------------------------------------
# Step 1. Import Libraries
# --------------------------------------------------
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# --------------------------------------------------
# Step 2. Load the Model and Tokenizer
# --------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to("cuda")

# --------------------------------------------------
# Step 3. Define the Prompt
# --------------------------------------------------
prompt = (
    "Write a detailed summary of the following paragraph:\n\n"
    "The Amazon Rainforest is the largest tropical rainforest in the world..."
)

# --------------------------------------------------
# Step 4. Tokenize the Input
# --------------------------------------------------
inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to("cuda")

# --------------------------------------------------
# Step 5. Generate the Summary
# --------------------------------------------------
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    min_length=40,
    num_beams=6,
    repetition_penalty=1.3,
    length_penalty=1.1,
    early_stopping=True
)

# --------------------------------------------------
# Step 6. Decode and Print the Result
# --------------------------------------------------
summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Summary:", summary)

# Listing 6-9: A Mini Chatbot Using a Transformer

Implements a simple conversational system with FLAN-T5 by rebuilding the recent conversation into a prompt and generating a reply from that context. The example illustrates the core loop behind chat systems: maintain context, tokenize it, generate a response, and append the new turn to history.

In [ ]:
# --------------------------------------------------
# Step 1. Install and Import Libraries
# --------------------------------------------------
!pip install -q transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --------------------------------------------------
# Step 2. Load the Model
# --------------------------------------------------
model_id = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# --------------------------------------------------
# Step 3. Initialize Conversation History
# --------------------------------------------------
history = []

# --------------------------------------------------
# Step 4. Build the Prompt Function
# --------------------------------------------------
def build_prompt(history, user_input):
    prompt = "The following is a conversation between a helpful AI Mini Chatbot and a user.\n"
    for turn in history:
        prompt += f"User: {turn['user']}\nMini Chatbot: {turn['bot']}\n"
    prompt += f"User: {user_input}\nMini Chatbot:"
    return prompt

# --------------------------------------------------
# Step 5. Start the Interactive Chat Loop
# --------------------------------------------------
print("FLAN-T5 Chatbot — type 'exit' to quit\n")

while True:
    user_input = input("You: ")

    if user_input.strip().lower() in ['bye', 'exit', 'quit']:
        print("Mini Chatbot: Bye.")
        break

    prompt = build_prompt(history, user_input)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=150
        )

    reply = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    print(f"Mini Chatbot: {reply}\n")

    history.append({"user": user_input, "bot": reply})
    if len(history) > 6:
        history = history[-6:]

# Listing 6-10: A Text-to-Video Generator Using a Transformer-Guided Diffusion Pipeline

Uses a pretrained text-to-video pipeline to generate short video clips from natural language prompts. A transformer-based text encoder converts the prompt into a semantic representation that guides diffusion-based frame generation, after which the frames are assembled into an MP4 video.

In [ ]:
# --------------------------------------------------
# Step 1. Install Dependencies
# --------------------------------------------------
!pip install -q diffusers transformers accelerate imageio[ffmpeg]

# --------------------------------------------------
# Step 2. Import Libraries
# --------------------------------------------------
import torch
import numpy as np
import imageio
from diffusers import TextToVideoSDPipeline
from IPython.display import HTML
from base64 import b64encode
import uuid

# --------------------------------------------------
# Step 3. Load the Text-to-Video Pipeline
# --------------------------------------------------
pipe = TextToVideoSDPipeline.from_pretrained("cerspense/zeroscope_v2_576w")
pipe.to("cuda")

# --------------------------------------------------
# Step 4. Define the Frame Sanitization Function
# --------------------------------------------------
def sanitize_frame(frame):
    if isinstance(frame, torch.Tensor):
        frame = frame.detach().cpu().numpy()
    if frame.dtype in [np.float32, np.float64]:
        frame = np.clip(frame, 0, 1)
        frame = (frame * 255).astype(np.uint8)
    elif not np.issubdtype(frame.dtype, np.uint8):
        frame = np.clip(frame, 0, 255).astype(np.uint8)
    if frame.shape[-1] > 3:
        frame = frame[..., :3]
    elif frame.shape[-1] == 1:
        frame = np.repeat(frame, 3, axis=-1)
    return frame

# --------------------------------------------------
# Step 5. Start the Interactive Generation Loop
# --------------------------------------------------
while True:
    prompt = input("\nEnter your text-to-video prompt (or type 'exit' to quit): ").strip()

    if prompt in ['exit', 'quit', 'bye']:
        print("Exiting. Thanks for using the text-to-video generator!")
        break

    print(f"\nGenerating video for prompt: '{prompt}'...")
    result = pipe(prompt, num_inference_steps=25).frames
    video_rgb_frames = [sanitize_frame(frame) for frame in result[0]]

    # Generate a unique filename
    video_path = f"/content/generated_{uuid.uuid4().hex[:8]}.mp4"
    imageio.mimsave(video_path, video_rgb_frames, fps=10)

    # Display the video inline
    mp4 = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"<video width=512 controls><source src='{data_url}' type='video/mp4'></video>"))